In [1]:
import os
import pandas as pd
import spatialdata as sd
import numpy as np
import glob
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed
import sys
sys.path.extend([
    "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1",
    "../../../scripts"]
)

import readwrite
cfg = readwrite.config()

def explode_df(df, c='morphological_features'):
    matrix = np.stack(df[c].values)
    col_names = [f'he_{i}' for i in range(matrix.shape[1])]
    return pd.DataFrame(matrix, index=df.index, columns=col_names)

def make_master_table(sdata):
    def delete_repeated_columns(df):
        return df.loc[:,~df.columns.duplicated()].copy()

    tile_coords_xenium_absolute = sdata.shapes['tile_coords_xenium_absolute'] 
    tile_coords_xenium_absolute = tile_coords_xenium_absolute.rename(columns={'geometry': 'xenium_tile_geometry'})
    tile_coords_xenium_absolute = delete_repeated_columns(tile_coords_xenium_absolute)
    tile_coords_hne_absolute = sdata.shapes['tile_coords_hne_absolute'] 
    tile_coords_hne_absolute = tile_coords_hne_absolute.rename(columns={'geometry': 'hne_tile_geometry'})
    metadata_table = sdata['features_adata'].obs

    metadata_table = metadata_table.merge(tile_coords_hne_absolute, on=["tile_id", "tissue_id"], how="inner")
    metadata_table = metadata_table.merge(tile_coords_xenium_absolute, on=["tile_id", "tissue_id"], how="inner")
    metadata_table['morphological_features'] = list(sdata.tables['features_adata'].X)

    cells_in_tile = sdata.shapes['cells_in_tile']
    cells_in_tile = cells_in_tile.rename(columns={"geometry": "cell_geometry"})
    cells_in_tile = cells_in_tile.drop(["index_right", 'component_and_cluster_and_lasso', "background_fraction"], axis=1)

    return cells_in_tile.merge(metadata_table, on=["tile_id", "tissue_id"], how="inner")

def get_cell_bulk_features(master_table):
    df = master_table.groupby("cell_id").agg({'morphological_features': 'mean'})
    df = explode_df(df)
    return df

def get_tile_bulk_features(master_table):
    df = master_table.groupby(["tile_id", "tissue_id"]).agg({'morphological_features': 'mean'})
    df = explode_df(df)
    return df

def get_organoid_bulk_features(master_table):
    df = master_table.groupby(["organoid_id"]).agg({'morphological_features': 'mean'})
    df = explode_df(df)
    return df

def load_sdata(file_path):
    def get_organoid_id(file_path):
        oid = os.path.basename(file_path).replace('_features.zarr', '')
        # remove redundant sample name prefix
        oid = 'proseg'+'_'.join(oid.split('_proseg')[1:])
        return oid

    organoid_id = get_organoid_id(file_path)
    sdata = sd.read_zarr(file_path)
    sdata.tables['features_adata'].obs['organoid_id'] = organoid_id
    return organoid_id, sdata, sdata.tables['features_adata']

def process_sdata(k, sdata, max_background_fraction=None):
    try:

        df_ = make_master_table(sdata)
        if max_background_fraction is not None:
            df_ = df_[df_['background_fraction'] < max_background_fraction]

        df_['tile_id'] = df_[['full_cell_id', 'tissue_id', 'tile_id']].astype(str).agg('_'.join, axis=1)

        df_tile_info = df_.set_index('full_cell_id')[['tile_id']]

        return (
            k,
            get_cell_bulk_features(df_),
            get_organoid_bulk_features(df_),
            get_tile_bulk_features(df_),
            df_tile_info,
            None
        )
    except Exception as e:
        return k, None, None, None, None, e

/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/norkin_organoid/.pixi/envs/cellcharter/lib/python3.12/site-packages/anndata/__init__.py:44: F

# Parameters

In [2]:
FEATURES_PATH = "/work/PRTNR/CHUV/DIR/rgottar1/spatial/env/lmcconn1/norkin_organoid/data/organoids_h&e/features_sdata"

# Read data

In [ ]:
sdata_paths = glob.glob(os.path.join(FEATURES_PATH, "*.zarr"))

sdatas_dict = {}

results = Parallel(n_jobs=-1, backend="threading")(
    delayed(load_sdata)(path) for path in tqdm(sdata_paths)
)

for organoid_id, sdata, adata_obj in results:
    sdatas_dict[organoid_id] = sdata

# ensure reproducible ordering
sdatas_dict = dict(sorted(sdatas_dict.items()))

In [ ]:
df_background = []
for k, v in sdatas_dict.items():
    df_background.append(v.shapes['tile_coords_xenium_absolute']['background_fraction'])
df_background = pd.concat(df_background)
df_background.hist(bins=500)

# Create aggregated tables

In [ ]:
max_background_fractions = np.arange(1, 10) / 10
base_output_path = Path(cfg['results_dir'] + "organoids_h&e")
base_output_path.mkdir(parents=True, exist_ok=True)
n_jobs = -1  

for max_frac in max_background_fractions:
    print(f"Processing {max_frac=}")

    # get table
    results = Parallel(
        n_jobs=n_jobs,
        backend="loky",
        verbose=0
    )(
        delayed(process_sdata)(k, sdata, max_frac)
        for k, sdata in tqdm(sdatas_dict.items())
    )

    # collect results
    df_cells = {}
    df_organoids = {}
    df_tiles = {}
    df_tiles_info = {}

    success_cnt = 0
    fail_cnt = 0

    for k, cell_df, org_df, tile_df, tile_info_df, err in results:
        if err is None:
            df_cells[k] = cell_df
            df_organoids[k] = org_df
            df_tiles[k] = tile_df
            df_tiles_info[k] = tile_info_df
            success_cnt += 1
        else:
            fail_cnt += 1
            print(f"{k} failed: {err}")

    print(f"{max_frac=} Success: {success_cnt}, Fail: {fail_cnt}")

    # ensure reproducible ordering
    df_cells = dict(sorted(df_cells.items()))
    df_organoids = dict(sorted(df_organoids.items()))
    df_tiles = dict(sorted(df_tiles.items()))
    df_tiles_info = dict(sorted(df_tiles_info.items()))

    # concat
    df_cells = pd.concat(df_cells)
    df_organoids = pd.concat(df_organoids)
    df_tiles = pd.concat(df_tiles)
    df_tiles_info = pd.concat(df_tiles_info)

    # save
    df_cells.to_parquet(base_output_path / f"cell_features_{max_frac=}.parquet")
    df_organoids.to_parquet(base_output_path / f"organoid_features_{max_frac=}.parquet")
    df_tiles.to_parquet(base_output_path / f"tile_features_{max_frac=}.parquet")
    df_tiles_info.to_parquet(base_output_path / f"tile_info_{max_frac=}.parquet")